In [90]:
import os
import pandas as pd

In [91]:
dir_path = r"C:\Users\glj523\Downloads\lanebarcode\lanebarcode"

In [92]:
file_names = os.listdir(dir_path)
df = pd.DataFrame(file_names)

In [93]:
df.rename(columns={0: 'filename'}, inplace=True)

In [94]:
for row in df.iterrows():
    split = row[1]['filename'].split('.')
    if split[1:] != ['lanebarcode', 'html']:
        raise ValueError(f"Unexpected file format: {row[1]['filename']}")
    else :
        df.at[row[0], 'run'] = split[0]

In [95]:
df['date'] = df['run'].apply(lambda x: x.split('_')[0])

In [96]:
df

,filename,run,date
0,20241106_A00706_0891_AHK27KDRX5.lanebarcode.html,20241106_A00706_0891_AHK27KDRX5,20241106
1,20241106_A00706_0892_BH5CTKDSX7.lanebarcode.html,20241106_A00706_0892_BH5CTKDSX7,20241106
2,20241108_A00706_0893_AH5CVTDSX7.lanebarcode.html,20241108_A00706_0893_AH5CVTDSX7,20241108
3,20241129_A00706_0894_BH5FLYDSX7.lanebarcode.html,20241129_A00706_0894_BH5FLYDSX7,20241129
4,20241129_A00706_0895_AHKVJGDRX5.lanebarcode.html,20241129_A00706_0895_AHKVJGDRX5,20241129
...,...,...,...
216,240926_A00706_0885_AH5FHKDSX7_TTS56_ITXBC.lane...,240926_A00706_0885_AH5FHKDSX7_TTS56_ITXBC,240926
217,241008_A00706_0886_AHMGK3DSXC_FASTQ.lanebarcod...,241008_A00706_0886_AHMGK3DSXC_FASTQ,241008
218,241011_A00706_0887_AHMFH5DSXC_HL8MP.lanebarcod...,241011_A00706_0887_AHMFH5DSXC_HL8MP,241011
219,241015_A00706_0889_AHVJTCDSXC_YLP56.lanebarcod...,241015_A00706_0889_AHVJTCDSXC_YLP56,241015


In [97]:
df['date'] = df['date'].apply(lambda x: x[2:] if len(x) == 8 else x)

In [98]:
df['machine'] = df['run'].apply(lambda x: x.split('_')[1])

In [99]:
df['run_no'] = df['run'].apply(lambda x: x.split('_')[2])

In [100]:
df['side'] = df['run'].apply(lambda x: x.split('_')[3][0])
df['flowcell'] = df['run'].apply(lambda x: x.split('_')[3][1:])


In [101]:
df.machine.apply(len).value_counts()

machine
6    221
Name: count, dtype: int64

In [102]:
df.astype(str).map(len).date.value_counts()

date
6    221
Name: count, dtype: int64

In [103]:
df

,filename,run,date,machine,run_no,side,flowcell
0,20241106_A00706_0891_AHK27KDRX5.lanebarcode.html,20241106_A00706_0891_AHK27KDRX5,241106,A00706,0891,A,HK27KDRX5
1,20241106_A00706_0892_BH5CTKDSX7.lanebarcode.html,20241106_A00706_0892_BH5CTKDSX7,241106,A00706,0892,B,H5CTKDSX7
2,20241108_A00706_0893_AH5CVTDSX7.lanebarcode.html,20241108_A00706_0893_AH5CVTDSX7,241108,A00706,0893,A,H5CVTDSX7
3,20241129_A00706_0894_BH5FLYDSX7.lanebarcode.html,20241129_A00706_0894_BH5FLYDSX7,241129,A00706,0894,B,H5FLYDSX7
4,20241129_A00706_0895_AHKVJGDRX5.lanebarcode.html,20241129_A00706_0895_AHKVJGDRX5,241129,A00706,0895,A,HKVJGDRX5
...,...,...,...,...,...,...,...
216,240926_A00706_0885_AH5FHKDSX7_TTS56_ITXBC.lane...,240926_A00706_0885_AH5FHKDSX7_TTS56_ITXBC,240926,A00706,0885,A,H5FHKDSX7
217,241008_A00706_0886_AHMGK3DSXC_FASTQ.lanebarcod...,241008_A00706_0886_AHMGK3DSXC_FASTQ,241008,A00706,0886,A,HMGK3DSXC
218,241011_A00706_0887_AHMFH5DSXC_HL8MP.lanebarcod...,241011_A00706_0887_AHMFH5DSXC_HL8MP,241011,A00706,0887,A,HMFH5DSXC
219,241015_A00706_0889_AHVJTCDSXC_YLP56.lanebarcod...,241015_A00706_0889_AHVJTCDSXC_YLP56,241015,A00706,0889,A,HVJTCDSXC


In [104]:
all(df.iloc[:, 2:].astype(str).map(len).nunique() == 1)

True

In [105]:
def find_valid_elements(path):
        path_split = path.split('_')
        res = []
        for ele in path_split:
            if len(ele)== 5 and not any(x.islower() for x in ele):
                res.append(ele)
        if len(res) > 0:
            return res
        else:
            return None

In [106]:
df['ttags'] = df.run.apply(find_valid_elements)

In [107]:
mask = (df.ttags.str.len() > 1) | df.ttags.isna()

In [108]:
helpr_path = r"C:\Users\glj523\Downloads\Copy of list_all_SeqRuns_2024-2025_edit251030JBT.xlsx"

In [109]:
helpr = pd.read_excel(helpr_path)

In [110]:
df['full_flowcell_id'] = df['side'] + df['flowcell']

In [111]:
helpr['full_flowcell_id'] = helpr['InstrumentSide'] + helpr['FlowCellID']

In [112]:
# We don't expect to have multiple ttags unless the RunSheetID contains 'xp'
mask = helpr['RunSheetID'].apply(lambda x: 'xp' in x.lower() if isinstance(x, str) else x)
helpr['RunSheetID_contains_xp'] = mask

In [114]:
mask = helpr['full_flowcell_id'].duplicated(keep=False)

In [115]:
helpr['duplicate_flowcell'] = mask

In [116]:
mask = helpr['duplicate_flowcell'] & ~helpr['RunSheetID_contains_xp']
helpr[mask].sort_values('full_flowcell_id').to_csv(r'Copy of list_all_SeqRuns_2024-2025_edit251030JBT_duplicated.csv', index=False)

In [117]:
helpr[mask].sort_values('full_flowcell_id')

,ID,BSSHRunID,WorkingID,RunSheetID,RunDate,InstrumentID,FlowCellID,InstrumentSide,SequencingOK,full_flowcell_id,RunSheetID_contains_xp,duplicate_flowcell
183,77,240227_A00706_0833_AH35MFDSX7,240221_BWDQD_Zaruhi,240226_Runsheet_S4_100PEDI_A00706_RunA,2024-02-27,A00706,H35MFDSX7,A,True,AH35MFDSX7,False,True
182,78,240227_A00706_0833_AH35MFDSX7,240221_BWDQD_Zaruhi,240507_Runsheet_S4_100PE_A00706_RunA,2024-02-27,A00706,H35MFDSX7,A,True,AH35MFDSX7,False,True
181,162,240227_A00706_0833_AH35MFDSX7,240221_BWDQD_Zaruhi,240226_Runsheet_S4_100PEDI_A00706_RunA,2024-02-27,A00706,H35MFDSX7,A,True,AH35MFDSX7,False,True
141,63,240709_A00706_0865_AH5FCGDSX7,240628_BBN2F_Maria,240709_S4FCA,2024-07-09,A00706,H5FCGDSX7,A,True,AH5FCGDSX7,False,True
140,181,240709_A00706_0865_AH5FCGDSX7,240628_BBN2F_Maria,240709_S4FCA,2024-07-09,A00706,H5FCGDSX7,A,True,AH5FCGDSX7,False,True
46,368,250630_A00706_0949_AHH72YDSXF,250502_THN45_Yucheng,250630_S4FCA,2025-06-30,A00706,HH72YDSXF,A,True,AHH72YDSXF,False,True
47,367,250630_A00706_0949_AHH72YDSXF,250502_THN45_Yucheng,250630_S4FCA,2025-06-30,A00706,HH72YDSXF,A,True,AHH72YDSXF,False,True
152,88,240515_A00706_0858_AHHGJVDSXC,240131_TUB01_Nikoline,240515_RunSheet_S4_150PEDI_A00706_RunA,2024-05-15,A00706,HHGJVDSXC,A,True,AHHGJVDSXC,False,True
151,159,240515_A00706_0858_AHHGJVDSXC,240131_E8KTD_Nikoline,240515_RunSheet_S4_150PEDI_A00706_RunA,2024-05-15,A00706,HHGJVDSXC,A,True,AHHGJVDSXC,False,True
127,195,240916_A00706_0879_AHHVLVDSXC,240821_I5R2U_Xihan,240916_S4FCA,2024-09-16,A00706,HHVLVDSXC,A,True,AHHVLVDSXC,False,True


In [86]:
mask = df['full_flowcell_id'].duplicated(keep=False)
df[mask].sort_values('full_flowcell_id').to_csv('duplicated_lanebarcodes.csv', index=False)

In [ ]:
merged_df = df.merge(helpr, on='full_flowcell_id', how='outer', indicator=True)

In [24]:
trans = {'left_only': 'Only found in lanebarcode filenames',
         'right_only': 'Only found in excel sheet',
         'both': 'Found in both lanebarcode filenames and excel sheet'}
merged_df['_merge'] = merged_df['_merge'].apply(lambda x: trans[x])
merged_df = merged_df.rename(columns={'_merge': 'flowcell_id_source'})  

In [25]:
from pathlib import Path
import sys
import os

def find_project_root():
    path = Path(os.getcwd()).resolve()
    while path != path.root:
        if (path / 'very_rootsy_file.txt').exists():
            return path
        path = path.parent
    return None  # Project root not found

project_root = find_project_root()

sys.path.append(str(project_root))

In [26]:
%env PGPASSWORD=5J8DhII0RRsPW1

env: PGPASSWORD=5J8DhII0RRsPW1


In [27]:
from constants.db_connections import ENGINE, ENGINE_READ_ONLY, SQL_ALCH_CONFIG, PSY_CONN
q = 'select * from flowcell'
db = pd.read_sql(q, ENGINE_READ_ONLY)
db['full_flowcell_id'] = db['flowcell_position'] + db['flowcell_id']
m3 = merged_df['full_flowcell_id'].isin(db['full_flowcell_id'].unique())
merged_df['in_db'] = m3

In [ ]:
mask = (merged_df['RunSheetID_contains_xp'] == False) & (merged_df.ttags.apply(lambda x: isinstance(x, list) and len(x) > 1))
merged_df['ttag_count_issue'] = mask

In [32]:
mask = (merged_df['in_db'] == False) & (merged_df['RunSheetID_contains_xp'] == False) & (merged_df['flowcell_id_source'] == 'Found in both lanebarcode filenames and excel sheet') & (merged_df['ttag_count_issue'] == False)
merged_df['upload_to_db'] = mask

In [33]:
mask = merged_df['upload_to_db']
upload_to_db = merged_df[mask]

Get all lanebarcode data to upload into one df

In [34]:
from constants.db_names.names import data
import lane_barcode_parser

fc = []
tpb = []

for file_name in upload_to_db.filename:
    file_path = os.path.join(dir_path, file_name)

    tube_tag_col_name = data.flowcell.library_pool_tag()
    read_len_col_name = data.flowcell.read_length()
    seq_machine_col_name = data.flowcell.sequencing_machine()
    seq_date_col_name = data.flowcell.sequencing_date()
    flowcell_pos_col_name = data.flowcell.flowcell_position()
    seq_run_col_name = data.flowcell.sequencing_run()


    # TODO: Make more general:
    sheet = pd.read_html(file_path, thousands=',', decimal='.')

    flowcell_data, top_unknown_barcodes = lane_barcode_parser.parse(df=sheet)

    fc.append(flowcell_data)
    tpb.append(top_unknown_barcodes)

In [35]:
flowcell_data = pd.concat(fc, ignore_index=True)
top_unknown_barcodes = pd.concat(tpb, ignore_index=True)

Get read length, sequencing date, tubetag, seq machine, seq run and flowcell pos merged in

In [38]:
upload_to_db

,filename,run,date,machine,run_no,side,flowcell,ttags,full_flowcell_id,ID,...,RunDate,InstrumentID,FlowCellID,InstrumentSide,SequencingOK,flowcell_id_source,in_db,RunSheetID_contains_xp,ttag_count_issue,upload_to_db
3,240219_A00706_0829_AH32TTDSX7_606A.lanebarcode...,240219_A00706_0829_AH32TTDSX7_606A,240219,A00706,0829,A,H32TTDSX7,None,AH32TTDSX7,8.0,...,2024-02-19,A00706,H32TTDSX7,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True
16,240212_A00706_0822_AH35MMDSX7_G8HU5.lanebarcod...,240212_A00706_0822_AH35MMDSX7_G8HU5,240212,A00706,0822,A,H35MMDSX7,[G8HU5],AH35MMDSX7,3.0,...,2024-02-12,A00706,H35MMDSX7,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True
18,240216_A00706_0826_AH3725DSX7_L8R4I.lanebarcod...,240216_A00706_0826_AH3725DSX7_L8R4I,240216,A00706,0826,A,H3725DSX7,[L8R4I],AH3725DSX7,5.0,...,2024-02-16,A00706,H3725DSX7,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True
19,240216_A00706_0826_AH3725DSX7_L8R4I_reDMPX.lan...,240216_A00706_0826_AH3725DSX7_L8R4I_reDMPX,240216,A00706,0826,A,H3725DSX7,[L8R4I],AH3725DSX7,5.0,...,2024-02-16,A00706,H3725DSX7,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True
20,240216_A00706_0826_AH3725DSX7_L8R4I_reDMPX_new...,240216_A00706_0826_AH3725DSX7_L8R4I_reDMPX_new,240216,A00706,0826,A,H3725DSX7,[L8R4I],AH3725DSX7,5.0,...,2024-02-16,A00706,H3725DSX7,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True
34,20241108_A00706_0893_AH5CVTDSX7.lanebarcode.html,20241108_A00706_0893_AH5CVTDSX7,241108,A00706,0893,A,H5CVTDSX7,None,AH5CVTDSX7,267.0,...,2024-11-08,A00706,H5CVTDSX7,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True
36,240801_A00706_0873_AH5CY7DSX7_E5VBT.lanebarcod...,240801_A00706_0873_AH5CY7DSX7_E5VBT,240801,A00706,0873,A,H5CY7DSX7,[E5VBT],AH5CY7DSX7,188.0,...,2024-08-01,A00706,H5CY7DSX7,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True
40,240621_A00706_0860_AH5F3TDSX7_L5WAS.lanebarcod...,240621_A00706_0860_AH5F3TDSX7_L5WAS,240621,A00706,0860,A,H5F3TDSX7,[L5WAS],AH5F3TDSX7,86.0,...,2024-06-21,A00706,H5F3TDSX7,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True
41,20250117_A00706_0910_AH5F73DSX7.lanebarcode.html,20250117_A00706_0910_AH5F73DSX7,250117,A00706,0910,A,H5F73DSX7,None,AH5F73DSX7,302.0,...,2025-01-17,A00706,H5F73DSX7,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True
44,240507_A00706_0857_AH5FC2DSX7_BWDQD.lanebarcod...,240507_A00706_0857_AH5FC2DSX7_BWDQD,240507,A00706,0857,A,H5FC2DSX7,[BWDQD],AH5FC2DSX7,174.0,...,2024-05-07,A00706,H5FC2DSX7,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True


In [42]:
mask = upload_to_db.flowcell.duplicated()
upload_to_db[mask]

,filename,run,date,machine,run_no,side,flowcell,ttags,full_flowcell_id,ID,...,RunDate,InstrumentID,FlowCellID,InstrumentSide,SequencingOK,flowcell_id_source,in_db,RunSheetID_contains_xp,ttag_count_issue,upload_to_db
19,240216_A00706_0826_AH3725DSX7_L8R4I_reDMPX.lan...,240216_A00706_0826_AH3725DSX7_L8R4I_reDMPX,240216,A00706,0826,A,H3725DSX7,[L8R4I],AH3725DSX7,5.0,...,2024-02-16,A00706,H3725DSX7,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True
20,240216_A00706_0826_AH3725DSX7_L8R4I_reDMPX_new...,240216_A00706_0826_AH3725DSX7_L8R4I_reDMPX_new,240216,A00706,0826,A,H3725DSX7,[L8R4I],AH3725DSX7,5.0,...,2024-02-16,A00706,H3725DSX7,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True
48,240709_A00706_0865_AH5FCGDSX7_BBN2F.lanebarcod...,240709_A00706_0865_AH5FCGDSX7_BBN2F,240709,A00706,0865,A,H5FCGDSX7,[BBN2F],AH5FCGDSX7,63.0,...,2024-07-09,A00706,H5FCGDSX7,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True
85,240515_A00706_0858_AHHGJVDSXC_Nikoline_E8KTD.l...,240515_A00706_0858_AHHGJVDSXC_Nikoline_E8KTD,240515,A00706,0858,A,HHGJVDSXC,[E8KTD],AHHGJVDSXC,88.0,...,2024-05-15,A00706,HHGJVDSXC,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True
163,240126_A00706_0811_AHTMW2DSX5_603A.lanebarcode...,240126_A00706_0811_AHTMW2DSX5_603A,240126,A00706,0811,A,HTMW2DSX5,None,AHTMW2DSX5,37.0,...,2024-01-26,A00706,HTMW2DSX5,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True
188,20250219_A00706_0914_AHVVLWDSXC.lanebarcode.html,20250219_A00706_0914_AHVVLWDSXC,250219,A00706,0914,A,HVVLWDSXC,None,AHVVLWDSXC,311.0,...,2025-02-19,A00706,HVVLWDSXC,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True
189,20250219_A00706_0914_AHVVLWDSXC_eDNAlib108.lan...,20250219_A00706_0914_AHVVLWDSXC_eDNAlib108,250219,A00706,0914,A,HVVLWDSXC,None,AHVVLWDSXC,316.0,...,2025-02-19,A00706,HVVLWDSXC,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True
190,20250219_A00706_0914_AHVVLWDSXC_eDNAlib108.lan...,20250219_A00706_0914_AHVVLWDSXC_eDNAlib108,250219,A00706,0914,A,HVVLWDSXC,None,AHVVLWDSXC,311.0,...,2025-02-19,A00706,HVVLWDSXC,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True
193,20250423_A00706_0929_AHVVN7DSXC.lanebarcode.html,20250423_A00706_0929_AHVVN7DSXC,250423,A00706,0929,A,HVVN7DSXC,None,AHVVN7DSXC,338.0,...,2025-04-23,A00706,HVVN7DSXC,A,True,Found in both lanebarcode filenames and excel ...,False,False,False,True
236,240724_A00706_0869_BH5F2GDSX7_PQFUO_Marina.lan...,240724_A00706_0869_BH5F2GDSX7_PQFUO_Marina,240724,A00706,0869,B,H5F2GDSX7,[PQFUO],BH5F2GDSX7,185.0,...,2024-07-24,A00706,H5F2GDSX7,B,True,Found in both lanebarcode filenames and excel ...,False,False,False,True


In [43]:
mask = merged_df.flowcell.duplicated()
merged_df[mask]

,filename,run,date,machine,run_no,side,flowcell,ttags,full_flowcell_id,ID,...,RunDate,InstrumentID,FlowCellID,InstrumentSide,SequencingOK,flowcell_id_source,in_db,RunSheetID_contains_xp,ttag_count_issue,upload_to_db
6,240229_A00706_0834_AH3332DSX7_XP_2PM2S_BYN4J_H...,240229_A00706_0834_AH3332DSX7_XP_2PM2S_BYN4J_H...,240229,A00706,0834,A,H3332DSX7,"[2PM2S, BYN4J, HYO5K, TYKZG]",AH3332DSX7,169.0,...,2024-02-29,A00706,H3332DSX7,A,True,Found in both lanebarcode filenames and excel ...,False,True,False,False
7,240229_A00706_0834_AH3332DSX7_XP_2PM2S_BYN4J_H...,240229_A00706_0834_AH3332DSX7_XP_2PM2S_BYN4J_H...,240229,A00706,0834,A,H3332DSX7,"[2PM2S, BYN4J, HYO5K, TYKZG]",AH3332DSX7,168.0,...,2024-02-29,A00706,H3332DSX7,A,True,Found in both lanebarcode filenames and excel ...,False,True,False,False
8,240229_A00706_0834_AH3332DSX7_XP_2PM2S_BYN4J_H...,240229_A00706_0834_AH3332DSX7_XP_2PM2S_BYN4J_H...,240229,A00706,0834,A,H3332DSX7,"[2PM2S, BYN4J, HYO5K, TYKZG]",AH3332DSX7,167.0,...,2024-02-29,A00706,H3332DSX7,A,True,Found in both lanebarcode filenames and excel ...,False,True,False,False
9,240229_A00706_0834_AH3332DSX7_XP_2PM2S_BYN4J_H...,240229_A00706_0834_AH3332DSX7_XP_2PM2S_BYN4J_H...,240229,A00706,0834,A,H3332DSX7,"[2PM2S, BYN4J, HYO5K, TYKZG]",AH3332DSX7,12.0,...,2024-02-29,A00706,H3332DSX7,A,True,Found in both lanebarcode filenames and excel ...,False,True,False,False
11,240227_A00706_0833_AH35MFDSX7_BWDQD.lanebarcod...,240227_A00706_0833_AH35MFDSX7_BWDQD,240227,A00706,0833,A,H35MFDSX7,[BWDQD],AH35MFDSX7,78.0,...,2024-02-27,A00706,H35MFDSX7,A,True,Found in both lanebarcode filenames and excel ...,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
341,20241211_A00706_0897_BHVWKVDSXC.lanebarcode.html,20241211_A00706_0897_BHVWKVDSXC,241211,A00706,0897,B,HVWKVDSXC,None,BHVWKVDSXC,281.0,...,2024-12-11,A00706,HVWKVDSXC,B,True,Found in both lanebarcode filenames and excel ...,False,True,False,False
342,20241211_A00706_0897_BHVWKVDSXC.lanebarcode.html,20241211_A00706_0897_BHVWKVDSXC,241211,A00706,0897,B,HVWKVDSXC,None,BHVWKVDSXC,280.0,...,2024-12-11,A00706,HVWKVDSXC,B,True,Found in both lanebarcode filenames and excel ...,False,True,False,False
343,20241211_A00706_0897_BHVWKVDSXC.lanebarcode.html,20241211_A00706_0897_BHVWKVDSXC,241211,A00706,0897,B,HVWKVDSXC,None,BHVWKVDSXC,279.0,...,2024-12-11,A00706,HVWKVDSXC,B,True,Found in both lanebarcode filenames and excel ...,False,True,False,False
348,230308_A00706_0708_BHWLFLDRX2_MortenA_TTJ8K_GI...,230308_A00706_0708_BHWLFLDRX2_MortenA_TTJ8K_GIQZB,230308,A00706,0708,B,HWLFLDRX2,"[TTJ8K, GIQZB]",BHWLFLDRX2,NaN,...,NaT,NaN,NaN,NaN,NaN,Only found in lanebarcode filenames,False,NaN,False,False


In [40]:
len(upload_to_db.flowcell.unique())

46

In [36]:
flowcell_data

,Lane,Project,Sample,Barcode sequence,PF Clusters,% of the lane,% Perfect barcode,% One mismatch barcode,Yield (Mbases),% PF Clusters,% >= Q30 bases,Mean Quality Score,Flowcell ID,Clusters (Raw) sum,Clusters(PF) sum,Yield (MBases) sum
0,1,240219-S4-2KESP-Lundbeck-606,018345T-0-LV2001740460-LV7008960548-srri9-E-ve...,TGCCGGTCAG+TTGTATCAGG,32950097,1.14,93.71,6.29,6656,100.0,90.26,35.24,H32TTDSX7,15320088576,11625755847,2348403
1,1,240219-S4-2KESP-Lundbeck-606,020943T-0-LV2001740498-LV7008957428-srri9-E-ve...,CACTCAATTC+CACAACTTAA,27782239,0.96,95.21,4.79,5612,100.0,91.07,35.41,H32TTDSX7,15320088576,11625755847,2348403
2,1,240219-S4-2KESP-Lundbeck-606,021187T-0-LV2001740142-LV7008960572-srri9-E-ve...,TCTCACACGC+TCACTGTCCG,31109882,1.08,96.48,3.52,6284,100.0,90.34,35.25,H32TTDSX7,15320088576,11625755847,2348403
3,1,240219-S4-2KESP-Lundbeck-606,100026T-0-LV2001741423-LV7008960611-srri9-E-ve...,GGACCAGTGG+TAGGCTCGCG,30484361,1.06,95.44,4.56,6158,100.0,89.95,35.18,H32TTDSX7,15320088576,11625755847,2348403
4,1,240219-S4-2KESP-Lundbeck-606,100334T-0-LV2001740006-LV7008960596-srri9-E-ve...,ATATGCATGT+CCAGGCACCA,31398564,1.09,95.16,4.84,6343,100.0,92.15,35.62,H32TTDSX7,15320088576,11625755847,2348403
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14071,4,COREX,LV7008865470-LV6000653908-SK2024-8,CCGAAGCGCT+ATAGTTAGCA,44316155,1.41,91.58,8.42,8952,100.0,92.54,35.7,HVVTHDSXC,15320088576,12518579994,2528753
14072,4,COREX,LV7008865471-LV6000653871-SK2024-5,GATCGGATAA+CGGCCGATAC,42779908,1.36,98.07,1.93,8642,100.0,91.48,35.47,HVVTHDSXC,15320088576,12518579994,2528753
14073,4,COREX,LV7008865472-LV6000653903-SK2024-3,GAGGTTAGAC+GCAATACATT,49678862,1.58,97.41,2.59,10035,100.0,93.57,35.9,HVVTHDSXC,15320088576,12518579994,2528753
14074,4,COREX,LV7008865558-LV6000653873-HH2024-1,TAGTGGAAGC+TATAGGTACT,45436718,1.44,96.78,3.22,9178,100.0,92.64,35.72,HVVTHDSXC,15320088576,12518579994,2528753
